# 2c — Join, features, and the model table

**Inputs:**

| File | Contents |
|---|---|
| `data/curated/taxi_airport_hourly.parquet` | Table A — the spine. 26,256 rows on `(pickup_date, pickup_hour, airport)`. |
| `data/curated/flights_hourly.parquet` | Table B — scheduled and realised arrivals at JFK and LGA, same grid. |
| `data/curated/flights_hourly_ewr.parquet` | Newark arrivals on `(date, hour)`. 13,128 rows. |
| `data/curated/flight_column_roles.json` | The leakage contract written by 2b. |
| `data/landing/weather/weather_*.parquet` | Meteostat hourly observations for JFK, LGA, EWR, and Central Park. |

**Outputs:**

| File | Contents |
|---|---|
| `data/curated/model_table.parquet` | One row per `(date, hour, airport)`, every feature, both targets, and the split. 26,256 rows. |
| `data/curated/model_table_roles.json` | Feature list, targets, and leak-if-unlagged columns. Read by notebook 4 instead of hardcoding. |
| `data/curated/shapes_model_table.json` | Headline figures quoted in the report. |

## What this notebook is building towards

Notebook 4 fits **two models with different jobs**, not two algorithms racing on one target:

| | Model 1 | Model 2 |
|---|---|---|
| Question | How many pickups in this airport-hour? | What is a pickup worth in this airport-hour? |
| Target | `n_pickups` | `mean_total` |
| Family | Poisson GLM, log link | Gradient-boosted trees |
| Nature | Parametric, count, interpretable coefficients | Non-parametric, continuous, captures interactions |

Their product is expected revenue per airport-hour, which is the surface the recommendations
actually stand on: *queue at LaGuardia at 22:00* is only advice if those trips pay. The
marking scheme awards three marks for **combining** the findings of two models, and a volume
model multiplied by a value model has something to combine.

One consequence has to be stated here rather than discovered later: `mean_total` is undefined
in the hours with no pickups, so Model 2 trains on a strictly smaller set than Model 1. Those
hours are overwhelmingly overnight, which means the value model is silent exactly where the
demand model says not to go. That is acceptable, but it is a limitation of the pair and Step 7
counts it.

## The contract

Table A is the spine and everything left-joins onto it. Its key columns are renamed from
`(pickup_date, pickup_hour)` to `(date, hour)` once, at the top, so that a single key name
runs through the rest of the pipeline. Every join is a left join onto 26,256 rows and every
join asserts that the count is still 26,256 afterwards — a join that changes the row count is
a bug, not a result.

**Nothing here is scaled, standardised, or centred.** That belongs inside the model fit in
notebook 4, fitted on the training split alone. Standardising across the whole table before
the split leaks the test distribution into training, and it is invisible unless someone asks.

In [1]:
"""Join the curated taxi, flight, and weather tables into the model table.

Table A (taxi airport-hours) is the spine. The flight, Newark, and weather
tables are left-joined onto it, temporal, holiday, and lag features are
derived, the leakage contract from notebook 2b is enforced, and the train/test
split is materialised as a column.

Writes `model_table.parquet` and the roles manifest that notebook 4 reads in
place of a hardcoded feature list.
"""

import json
import math
import sys
from pathlib import Path

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

sys.path.append(str(Path("..") / "scripts"))
from spark_utils import (  # noqa: E402
    ARRIVAL_AIRPORTS,
    create_spark_session,
    hour_spine,
)

# --- Paths -----------------------------------------------------------------
# Notebook is expected to run from `notebooks/`.
PROJECT_ROOT = Path("..").resolve()
CURATED_DIR = PROJECT_ROOT / "data" / "curated"
WEATHER_DIR = PROJECT_ROOT / "data" / "landing" / "weather"

# --- Study window ----------------------------------------------------------
# Identical to notebooks 2a and 2b. Inclusive of the start, exclusive of the end.
WINDOW_START = "2023-01-01"
WINDOW_END = "2024-07-01"
WINDOW_DAYS = 547
EXPECTED_ROWS = WINDOW_DAYS * 24 * len(ARRIVAL_AIRPORTS)  # 26,256
EXPECTED_SITE_HOURS = WINDOW_DAYS * 24                    # 13,128 per weather site

# --- Split -----------------------------------------------------------------
# Train on the calendar year 2023, test on January–June 2024. The split is
# forward in time: the specification requires predictions to use future data,
# so no shuffled or k-fold scheme is admissible here.
TRAIN_END = "2024-01-01"      # exclusive
EXPECTED_TRAIN_ROWS = 365 * 24 * len(ARRIVAL_AIRPORTS)  # 17,520
EXPECTED_TEST_ROWS = 182 * 24 * len(ARRIVAL_AIRPORTS)   # 8,736

# --- Lags ------------------------------------------------------------------
LAG_WEEK_HOURS = 168   # same hour, one week earlier
LAG_DAY_HOURS = 24     # same hour, one day earlier
LAG_HOUR = 1           # the previous clock hour
TRAILING_DAYS = 7      # trailing mean of the same hour, over this many days

# --- Weather gap filling ---------------------------------------------------
# Two policies, because two kinds of quantity. Temperature, humidity, pressure
# and wind are smooth and strongly autocorrelated, so carrying the last
# observation forward for a few hours is defensible. Precipitation and the
# condition code are event-like — it raining an hour ago is not evidence that
# it is raining now — so they are carried forward one hour at most and left
# null beyond that.
MAX_FILL_SMOOTH_HOURS = 3
MAX_FILL_EVENT_HOURS = 1

In [2]:
spark = create_spark_session(app_name="MAST30034 — join and features")
spark.sparkContext.setLogLevel("WARN")
spark.version

your 131072x1 screen size is bogus. expect trouble
26/08/16 20:30:21 WARN Utils: Your hostname, iphone resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/16 20:30:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/16 20:30:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

## Step 0 — Load and align

The rename of Table A's key columns happens here and nowhere else. Doing it once at load
means no later step has to remember which table calls the key what, and the three joins
below can all be written on `(date, hour, airport)`.

In [3]:
table_a = (
    spark.read.parquet(str(CURATED_DIR / "taxi_airport_hourly.parquet"))
    .withColumnRenamed("pickup_date", "date")
    .withColumnRenamed("pickup_hour", "hour")
)
table_b = spark.read.parquet(str(CURATED_DIR / "flights_hourly.parquet"))
table_ewr = spark.read.parquet(str(CURATED_DIR / "flights_hourly_ewr.parquet"))

with open(CURATED_DIR / "flight_column_roles.json") as handle:
    flight_roles = json.load(handle)

# The spine defines the row count for everything that follows, so it is checked
# before anything is joined to it rather than after.
taxi_rows = table_a.count()
assert taxi_rows == EXPECTED_ROWS, (
    f"Table A has {taxi_rows:,} rows, expected {EXPECTED_ROWS:,}"
)
assert table_b.count() == EXPECTED_ROWS, "Table B is not on the same grid as Table A"
assert table_ewr.count() == EXPECTED_SITE_HOURS, (
    "Newark table is not a complete hourly grid"
)

print(f"Table A  {taxi_rows:,} rows  ×  {len(table_a.columns)} columns")
print(f"Table B  {EXPECTED_ROWS:,} rows  ×  {len(table_b.columns)} columns")
print(f"Newark   {EXPECTED_SITE_HOURS:,} rows  ×  {len(table_ewr.columns)} columns")

Table A  26,256 rows  ×  20 columns
Table B  26,256 rows  ×  14 columns
Newark   13,128 rows  ×  7 columns


## Step 1 — The joins, one assertion each

| Table | Key | Rows after |
|---|---|---|
| `flights_hourly` | `(date, hour, airport)` | 26,256 |
| `flights_hourly_ewr` | `(date, hour)`, broadcast across both airports | 26,256 |
| `weather` | `(obs_date, obs_hour, site)` → `(date, hour, airport)` | 26,256 |

`join_checked` asserts the count after each one. That single line catches the three bugs
that would otherwise surface as a wrong coefficient in notebook 4 rather than as a failure
here: a duplicated daylight-saving hour on the right-hand side, a many-to-one join that was
supposed to be one-to-one, and a key whose type does not match (a `string` date against a
`date` date joins to nothing and quietly fills the column with nulls).

It also checks the right-hand side for duplicate keys *before* joining, which localises the
fault. A row count that grew tells you a join went wrong; a duplicate-key check tells you
which table is wrong and how many rows of it.

The Newark table is deliberately keyed on `(date, hour)` alone, so its columns are broadcast
identically to the JFK and LGA rows of the same hour. That is intended: Newark is citywide
context, not an airport-specific measurement.

In [4]:
def join_checked(
    left: DataFrame,
    right: DataFrame,
    on: list,
    expected_rows: int,
    label: str,
) -> DataFrame:
    """Left-join two frames and fail immediately if the row count changes.

    Checks the right-hand side for duplicate keys before joining, so that a
    many-to-one join is reported against the table that caused it rather than
    as an unexplained row count downstream.

    Args:
        left: The spine. Its row count must be preserved.
        right: Frame to join on. Must be unique on ``on``.
        on: Join key columns, present in both frames.
        expected_rows: Row count the result must have.
        label: Name used in the printed confirmation and failure messages.

    Returns:
        The joined frame.

    Raises:
        AssertionError: If the right-hand side has duplicate keys, or if the
            join changed the row count.
    """
    right_rows = right.count()
    unique_keys = right.dropDuplicates(on).count()
    assert right_rows == unique_keys, (
        f"{label}: right-hand side has {right_rows - unique_keys:,} duplicate keys "
        f"on {on} and would multiply rows"
    )

    joined = left.join(right, on=on, how="left")
    actual = joined.count()
    assert actual == expected_rows, (
        f"{label}: {actual:,} rows after join, expected {expected_rows:,}"
    )

    print(f"{label:<28} {actual:,} rows  ×  {len(joined.columns)} columns")
    return joined


model_table = join_checked(
    table_a, table_b, ["date", "hour", "airport"], EXPECTED_ROWS, "+ flights (JFK, LGA)"
)
model_table = join_checked(
    model_table, table_ewr, ["date", "hour"], EXPECTED_ROWS, "+ flights (EWR)"
)

+ flights (JFK, LGA)         26,256 rows  ×  31 columns
+ flights (EWR)              26,256 rows  ×  36 columns


## Step 2 — Weather

Weather is the one input that has not been inspected yet: 2a and 2b each cleaned their own
source, but the Meteostat files come straight from `download.py` into this notebook. It gets
the same treatment the other two sources got, in the same order — look at what is missing,
decide what to do about it, then derive.

Three things are known in advance from the download script's docstring and are confirmed
rather than assumed below:

- `snow` is snow *depth*, which US METAR stations rarely report hourly, so it is expected to
  be almost entirely null. Snowfall is derived from the condition code instead.
- `wpgt` (peak gust) is sparsely reported for the same reason.
- `model=False` was passed to Meteostat, so a gap in the record is a genuine gap rather than
  numerical-model output dressed as an observation. Every null below is therefore a real
  missing measurement and can be counted honestly.

In [6]:
from pathlib import Path

print(f"Found {len(weather_paths)} weather files\n")

bad_files = []

for p in weather_paths:
    try:
        df_test = spark.read.parquet(p)

        # Force Spark to actually read some data
        df_test.limit(1).collect()

        print(f"OK     {Path(p).name}")

    except Exception as e:
        print(f"\nFAILED {Path(p).name}")
        print(str(e)[-4000:])
        bad_files.append(p)

print("\nBad files:")
for p in bad_files:
    print(p)

Found 4 weather files


FAILED weather_ewr.parquet


26/08/16 20:32:42 ERROR Executor: Exception in task 0.0 in stage 40.0 (TID 27)
org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1826)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:206)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:283)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:224)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertField(ParquetSchemaConverter.scala:187)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertInternal

java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:840)
Caused by: org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1826)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:206)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:283)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:224)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertField(

26/08/16 20:32:42 ERROR Executor: Exception in task 0.0 in stage 41.0 (TID 28)
org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1826)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:206)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:283)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:224)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertField(ParquetSchemaConverter.scala:187)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertInternal


FAILED weather_lga.parquet
java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:840)
Caused by: org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1826)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:206)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:283)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:224)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkS

In [5]:
weather_paths = sorted(str(path) for path in WEATHER_DIR.glob("weather_*.parquet"))
assert weather_paths, (
    f"No weather files in {WEATHER_DIR}. Run: "
    "python scripts/download.py --skip-taxi --skip-bts"
)

weather_raw = (
    spark.read.parquet(*weather_paths)
    .withColumn("obs_date", F.col("obs_date").cast("date"))
    .withColumn("obs_hour", F.col("obs_hour").cast("int"))
)

WEATHER_MEASURES = ["temp", "rhum", "prcp", "snow", "wspd", "wpgt", "pres", "coco"]

# Rows retrieved per site, against the 13,128 hours the window contains. A
# shortfall here is a *missing row*, which the spine below restores as a null;
# the null rates in the next cell are missing *values* within rows that exist.
# The two are different failures and are reported separately.
(
    weather_raw
    .groupBy("site")
    .agg(
        F.count("*").alias("rows"),
        F.min("obs_date").alias("first_date"),
        F.max("obs_date").alias("last_date"),
    )
    .withColumn("missing_hours", F.lit(EXPECTED_SITE_HOURS) - F.col("rows"))
    .orderBy("site")
    .show(truncate=False)
)

26/08/16 20:30:29 ERROR Executor: Exception in task 0.0 in stage 39.0 (TID 26)
org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1826)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:206)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:283)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:224)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertField(ParquetSchemaConverter.scala:187)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertInternal

Py4JJavaError: An error occurred while calling o65.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 39.0 failed 1 times, most recent failure: Lost task 0.0 in stage 39.0 (TID 26) (10.255.255.254 executor driver): org.apache.spark.sql.AnalysisException: Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false)).
	at org.apache.spark.sql.errors.QueryCompilationErrors$.illegalParquetTypeError(QueryCompilationErrors.scala:1826)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.illegalType$1(ParquetSchemaConverter.scala:206)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.$anonfun$convertPrimitiveField$2(ParquetSchemaConverter.scala:283)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.convertPrimitiveField(ParquetSchemaConverter.scala:224)
	at org.apache.spark.sql.execution.datasources.parquet.Parquet

In [ ]:
# Per-column null rates, per site. Printed before any decision is taken about
# filling, because the decision depends on these numbers and the report has to
# quote them.
null_rates = weather_raw.groupBy("site").agg(
    *[
        F.round(F.avg(F.col(column).isNull().cast("double")), 4).alias(column)
        for column in WEATHER_MEASURES
    ]
)
null_rates.orderBy("site").show(truncate=False)

### What is kept, and how gaps are filled

`snow` and `wpgt` are **dropped**. The null rates above show they are unusable, and an
almost-entirely-null column is worse than no column: it survives into a model as a mostly
imputed constant carrying whatever the imputation implied. Snowfall is instead derived from
the `coco` condition code, which is populated, and from `prcp`.

The remaining measures are filled by carrying the last observation forward, under two
policies and a strict lookback limit:

| Columns | Lookback | Reasoning |
|---|---|---|
| `temp`, `rhum`, `pres`, `wspd` | 3 hours | Smooth and strongly autocorrelated. The temperature three hours ago is a good estimate of the temperature now; a gap this short cannot hide a front passing through. |
| `prcp`, `coco` | 1 hour | Event-like. Rain an hour ago is weak evidence of rain now and rain three hours ago is almost none, so carrying these forward would manufacture precipitation that did not happen. |

The fill is **positional** — `rowsBetween(-3, 0)` over the site's rows — which is only
equivalent to a three-*hour* lookback if every hour is present exactly once. A complete spine
per site is therefore built first, exactly as 2a and 2b did for their own grids, and the
observations are left-joined onto it. That also converts a missing row into a null that the
fill can see, rather than a hole that it silently steps over.

Two flags are carried out of this step. `weather_imputed` marks an hour whose temperature was
filled rather than observed; `weather_missing` marks one that is still null after filling, so
notebook 4 can decide what to do about it rather than discover it in a fitted coefficient.

In [ ]:
SMOOTH_COLUMNS = ["temp", "rhum", "pres", "wspd"]
EVENT_COLUMNS = ["prcp", "coco"]
KEPT_COLUMNS = SMOOTH_COLUMNS + EVENT_COLUMNS

sites = weather_raw.select("site").distinct()
weather_spine = (
    hour_spine(spark, WINDOW_START, WINDOW_END, date_col="obs_date", hour_col="obs_hour")
    .crossJoin(sites)
)

weather = weather_spine.join(
    weather_raw.select("site", "obs_date", "obs_hour", *KEPT_COLUMNS),
    ["site", "obs_date", "obs_hour"],
    how="left",
)

site_hours = weather.groupBy("site").count().collect()
for row in site_hours:
    assert row["count"] == EXPECTED_SITE_HOURS, (
        f"{row['site']} has {row['count']:,} hours after the spine join"
    )
print(f"{len(site_hours)} sites × {EXPECTED_SITE_HOURS:,} hours")

# Positional carry-forward, bounded. `rowsBetween(-n, 0)` takes the last
# non-null value within the previous n rows, which the completed spine makes
# equal to the previous n hours.
def carry_forward(column: str, hours: int):
    """Last observation carried forward over a bounded window of rows.

    Args:
        column: Column to fill.
        hours: Maximum number of hours to look back. The completed spine makes
            one row equal to one hour, so this is a row count.

    Returns:
        A Column holding the filled value.
    """
    window = (
        Window.partitionBy("site")
        .orderBy("obs_date", "obs_hour")
        .rowsBetween(-hours, 0)
    )
    return F.last(F.col(column), ignorenulls=True).over(window)


weather = weather.withColumn("temp_observed", F.col("temp").isNotNull())
for column in SMOOTH_COLUMNS:
    weather = weather.withColumn(column, carry_forward(column, MAX_FILL_SMOOTH_HOURS))
for column in EVENT_COLUMNS:
    weather = weather.withColumn(column, carry_forward(column, MAX_FILL_EVENT_HOURS))

weather = (
    weather
    .withColumn(
        "weather_imputed", ~F.col("temp_observed") & F.col("temp").isNotNull()
    )
    .withColumn("weather_missing", F.col("temp").isNull())
    .drop("temp_observed")
)

weather.groupBy("site").agg(
    F.sum(F.col("weather_imputed").cast("int")).alias("hours_filled"),
    F.sum(F.col("weather_missing").cast("int")).alias("hours_still_missing"),
).orderBy("site").show()

### The condition code

`coco` is a Meteostat condition code, not a quantity: code 14 is light snowfall and code 7 is
light rain, and their difference is a category rather than seven units of anything. Fed to a
model as an integer it would be read as an ordering, so it is bucketed into five conditions
and the raw code is retained only for traceability.

`is_precip` and `is_snow` are derived from `coco` **and** `prcp` rather than from the dropped
`snow` column, per the caveat in `download.py`. Freezing rain and sleet are grouped with snow
rather than with rain, because for this study the relevant distinction is whether the ground
is frozen — that is what closes runways and lengthens taxi queues, not whether the water
arriving is technically liquid.

In [ ]:
# Meteostat condition codes, grouped. Reference:
# https://dev.meteostat.net/formats.html#weather-condition-codes
CONDITION_GROUPS = {
    "clear": [1, 2],                              # clear, fair
    "cloud": [3, 4],                              # cloudy, overcast
    "fog": [5, 6],                                # fog, freezing fog
    "rain": [7, 8, 9, 17, 18, 23, 25, 26, 27],    # rain, showers, thunderstorm
    # Frozen: snow, sleet, freezing rain, hail.
    "snow": [10, 11, 12, 13, 14, 15, 16, 19, 20, 21, 22, 24],
}
FROZEN_CODES = CONDITION_GROUPS["snow"]
WET_CODES = CONDITION_GROUPS["rain"] + FROZEN_CODES

condition = F.lit(None).cast("string")
for name, codes in CONDITION_GROUPS.items():
    condition = F.when(F.col("coco").isin(codes), F.lit(name)).otherwise(condition)

weather = (
    weather
    .withColumnRenamed("temp", "temp_c")
    .withColumnRenamed("rhum", "rhum_pct")
    .withColumnRenamed("prcp", "prcp_mm")
    .withColumnRenamed("wspd", "wspd_kmh")
    .withColumnRenamed("pres", "pres_hpa")
    .withColumn("coco_code", F.col("coco").cast("int"))
    .withColumn("condition", condition)
    # Either the code says water was falling or the gauge measured some. Null
    # only where both inputs are null, so an unobserved hour stays unobserved
    # rather than being recorded as dry.
    .withColumn(
        "is_precip",
        F.when(
            F.col("coco_code").isNotNull() | F.col("prcp_mm").isNotNull(),
            F.coalesce(F.col("coco_code").isin(WET_CODES), F.lit(False))
            | F.coalesce(F.col("prcp_mm") > 0, F.lit(False)),
        ),
    )
    .withColumn(
        "is_snow",
        F.when(
            F.col("coco_code").isNotNull(),
            F.col("coco_code").isin(FROZEN_CODES),
        ),
    )
    .drop("coco")
)

# Any code outside the groups above would fall through as a null `condition`
# while `coco_code` is populated. Reported rather than asserted: an unknown
# code is a data-quality note, not a reason to halt a marker's re-run.
unmapped = weather.where(
    F.col("coco_code").isNotNull() & F.col("condition").isNull()
).count()
print(f"Condition codes outside the documented groups: {unmapped}")

weather.groupBy("site", "condition").count().orderBy("site", F.desc("count")).show(20)

### Central Park as a cross-check

Four sites were downloaded. Only two are joined: a driver queueing at JFK is exposed to the
weather at JFK, and feeding both the airport series and the citywide series to the model
would supply two noisy measurements of one thing and invite it to split a coefficient
between them.

Central Park still earns its download as an independent check. If the JFK and LaGuardia
series were mis-keyed — the wrong station resolved, or a timezone conversion off by hours —
they would disagree with a station eight miles away far more than two New York airports
plausibly can. Newark is retained on disk for the same reason and is not joined either; its
flight columns are already in the table, and its weather would be a third measurement of the
same system.

In [ ]:
# Hourly temperature spread across sites: the maximum minus the minimum reading
# in the same hour. Two airport stations and Central Park should agree closely.
spread = (
    weather
    .where(F.col("temp_c").isNotNull())
    .groupBy("obs_date", "obs_hour")
    .agg(
        F.max("temp_c").alias("max_temp"),
        F.min("temp_c").alias("min_temp"),
        F.count("*").alias("sites_reporting"),
    )
    .where(F.col("sites_reporting") == len(site_hours))
    .withColumn("spread_c", F.col("max_temp") - F.col("min_temp"))
)

spread.agg(
    F.count("*").alias("hours_compared"),
    F.round(F.avg("spread_c"), 2).alias("mean_spread_c"),
    F.round(F.percentile_approx("spread_c", 0.95), 2).alias("p95_spread_c"),
    F.round(F.max("spread_c"), 2).alias("max_spread_c"),
).show()

In [ ]:
# Join the two airport series. The site code and the airport code are the same
# strings by construction in `download.py`, which is what makes this a rename
# rather than a lookup — but it is asserted, because a silent mismatch would
# fill every weather column with nulls instead of failing.
weather_airports = (
    weather
    .where(F.col("site").isin(list(ARRIVAL_AIRPORTS)))
    .withColumnRenamed("site", "airport")
    .withColumnRenamed("obs_date", "date")
    .withColumnRenamed("obs_hour", "hour")
)
assert (
    {row["airport"] for row in weather_airports.select("airport").distinct().collect()}
    == set(ARRIVAL_AIRPORTS)
), "Weather site codes do not match the airport codes used by Tables A and B"

model_table = join_checked(
    model_table,
    weather_airports,
    ["date", "hour", "airport"],
    EXPECTED_ROWS,
    "+ weather (JFK, LGA)",
).cache()

## Step 3 — Temporal and holiday features

Hour of day, day of week, and month are the calendar structure the models need in order to
attribute anything to the flight schedule or the weather rather than to the fact that it was
a Tuesday afternoon.

`day_of_week` uses the ISO convention (1 = Monday) rather than Spark's `dayofweek`
(1 = Sunday), so that the weekend is `{6, 7}` and reads as a range. It is a **categorical**
integer, not a quantity — Wednesday is not three of anything — and the roles manifest
declares it as such so notebook 4 indexes rather than regresses on it.

Cyclical `hour_sin` and `hour_cos` are added for the Poisson GLM, which otherwise cannot
express that hour 23 and hour 0 are adjacent without twenty-four dummy variables. The trees
do not need them and will ignore them.

**Holidays are hardcoded, deliberately.** A `holidays` package dependency would have to
install and resolve identically on a marker's machine for the pipeline to reproduce, and it
would silently change the feature if its rules were updated. Twenty-four dates for eighteen
months is a constant with a citation, auditable in place:
<https://www.opm.gov/policy-data-oversight/pay-leave/federal-holidays/>

Observed dates are listed alongside actual ones where they differ — 1 January 2023 fell on a
Sunday and was observed on the Monday, 11 November 2023 on a Saturday and observed on the
Friday — because air travel responds to the long weekend, which is the observed date, while
the holiday itself is the day people are travelling home from.

The adjacent days are not decoration. The Sunday after Thanksgiving is routinely the busiest
day of the year in US aviation, and Christmas Eve and New Year's Eve behave unlike any
ordinary day in December. Flagging them lets the models treat those days as their own regime
instead of averaging them into a residual.

In [ ]:
# US federal holidays falling inside the study window, plus the adjacent travel
# days that behave unlike ordinary days at an airport. `kind` separates the two
# so the report can say which is which.
HOLIDAYS = [
    # (date, name, kind)
    ("2023-01-01", "New Year's Day", "federal"),
    ("2023-01-02", "New Year's Day (observed)", "federal"),
    ("2023-01-16", "Martin Luther King Jr. Day", "federal"),
    ("2023-02-20", "Washington's Birthday", "federal"),
    ("2023-05-29", "Memorial Day", "federal"),
    ("2023-06-19", "Juneteenth", "federal"),
    ("2023-07-04", "Independence Day", "federal"),
    ("2023-09-04", "Labor Day", "federal"),
    ("2023-10-09", "Columbus Day", "federal"),
    ("2023-11-10", "Veterans Day (observed)", "federal"),
    ("2023-11-11", "Veterans Day", "federal"),
    ("2023-11-22", "Thanksgiving eve", "adjacent"),
    ("2023-11-23", "Thanksgiving Day", "federal"),
    ("2023-11-24", "Day after Thanksgiving", "adjacent"),
    ("2023-11-26", "Sunday after Thanksgiving", "adjacent"),
    ("2023-12-24", "Christmas Eve", "adjacent"),
    ("2023-12-25", "Christmas Day", "federal"),
    ("2023-12-26", "Day after Christmas", "adjacent"),
    ("2023-12-31", "New Year's Eve", "adjacent"),
    ("2024-01-01", "New Year's Day", "federal"),
    ("2024-01-15", "Martin Luther King Jr. Day", "federal"),
    ("2024-02-19", "Washington's Birthday", "federal"),
    ("2024-05-27", "Memorial Day", "federal"),
    ("2024-06-19", "Juneteenth", "federal"),
]

# One row per date, or the join below would multiply the table.
assert len({entry[0] for entry in HOLIDAYS}) == len(HOLIDAYS), "Duplicate holiday date"
assert all(WINDOW_START <= entry[0] < WINDOW_END for entry in HOLIDAYS), (
    "A holiday falls outside the study window"
)

holidays = (
    spark.createDataFrame(HOLIDAYS, ["holiday_date", "holiday_name", "holiday_kind"])
    .withColumn("date", F.to_date("holiday_date"))
    .drop("holiday_date")
)

model_table = join_checked(
    model_table, F.broadcast(holidays), ["date"], EXPECTED_ROWS, "+ holidays"
)

In [ ]:
# ISO day of week: Spark's `dayofweek` is 1 = Sunday, so it is rotated to
# 1 = Monday and the weekend becomes a contiguous range.
iso_dow = ((F.dayofweek("date") + 5) % 7) + 1

model_table = (
    model_table
    .withColumn("day_of_week", iso_dow)
    .withColumn("day_name", F.date_format("date", "EEE"))
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .withColumn("is_weekend", F.col("day_of_week") >= 6)
    .withColumn("is_holiday", F.coalesce(F.col("holiday_kind") == "federal", F.lit(False)))
    .withColumn(
        "is_holiday_adjacent",
        F.coalesce(F.col("holiday_kind") == "adjacent", F.lit(False)),
    )
    # Hour as an angle, so that 23:00 and 00:00 are adjacent to the GLM.
    .withColumn("hour_sin", F.sin(2 * math.pi * F.col("hour") / 24))
    .withColumn("hour_cos", F.cos(2 * math.pi * F.col("hour") / 24))
    .drop("holiday_kind")
)

model_table.groupBy("is_holiday", "is_holiday_adjacent").agg(
    F.countDistinct("date").alias("days"),
    F.round(F.avg("n_pickups"), 1).alias("mean_pickups_per_hour"),
).orderBy(F.desc("is_holiday"), F.desc("is_holiday_adjacent")).show()

## Step 4 — Lags

**Why a positional lag is legitimate here.** `F.lag(column, 168)` steps back 168 *rows*, not
168 hours. On a raw aggregation, where hours with no pickups produce no row, those are
different things and the feature would be silently wrong. The spine built in 2a guarantees
every airport has exactly 24 rows per day with no gaps, so 168 rows is exactly seven days of
wall-clock time — including across the daylight-saving boundaries, where the grid keeps 24
wall-clock hours per day by construction. The completeness of the spine is what makes this
step correct, which is worth one sentence in the report.

Both targets get their own history, because fare composition is at least as weekly-periodic
as volume: an 08:00 Monday at JFK is a different mix of Manhattan flat fares and short
Queens runs from a 22:00 Saturday, and last week's 08:00 Monday is the best guide to it.

| Feature | For | Meaning |
|---|---|---|
| `pickups_lag_168h` | Model 1 | Same hour, one week ago. Also the benchmark below. |
| `pickups_lag_24h` | Model 1 | Same hour, yesterday. |
| `pickups_same_hour_7d` | Model 1 | Trailing mean of the same hour over the previous seven days. |
| `mean_total_lag_168h` | Model 2 | Same hour, one week ago. |
| `mean_total_same_hour_7d` | Model 2 | Trailing mean of the same hour over the previous seven days. |

The trailing means use a window partitioned by `(airport, hour)` and ordered by date, framed
`rowsBetween(-7, -1)`. Each such partition holds exactly one row per date, so seven rows is
seven days, and the frame stops at `-1` so the window can never see the current row.

The first week of 2023 has no seven-day history: 7 days × 24 hours × 2 airports = **336 rows**
come out null. They are kept and flagged `has_full_lags` rather than deleted, so notebook 3's
distribution work still sees the whole window and only the model fit excludes them.

In [ ]:
# Ordered over the complete grid, per airport, so a positional lag is a
# wall-clock lag. Never framed forwards.
by_hour = Window.partitionBy("airport").orderBy("date", "hour")
same_hour_history = (
    Window.partitionBy("airport", "hour")
    .orderBy("date")
    .rowsBetween(-TRAILING_DAYS, -1)
)

model_table = (
    model_table
    # --- Model 1: volume history -------------------------------------------
    .withColumn("pickups_lag_168h", F.lag("n_pickups", LAG_WEEK_HOURS).over(by_hour))
    .withColumn("pickups_lag_24h", F.lag("n_pickups", LAG_DAY_HOURS).over(by_hour))
    .withColumn("pickups_lag_1h", F.lag("n_pickups", LAG_HOUR).over(by_hour))
    .withColumn("pickups_same_hour_7d", F.avg("n_pickups").over(same_hour_history))
    # --- Model 2: value history --------------------------------------------
    # `mean_total` is null in hours with no pickups, and `F.avg` ignores nulls,
    # so the trailing mean is taken over the days in the window that had trips
    # in this hour rather than being nulled by one quiet night.
    .withColumn("mean_total_lag_168h", F.lag("mean_total", LAG_WEEK_HOURS).over(by_hour))
    .withColumn(
        "mean_total_same_hour_7d", F.avg("mean_total").over(same_hour_history)
    )
    .withColumn("has_full_lags", F.col("pickups_lag_168h").isNotNull())
)

incomplete = model_table.where(~F.col("has_full_lags")).count()
expected_incomplete = TRAILING_DAYS * 24 * len(ARRIVAL_AIRPORTS)
assert incomplete == expected_incomplete, (
    f"{incomplete:,} rows lack a weekly lag, expected {expected_incomplete:,}"
)
print(f"{incomplete:,} rows without a full week of history (the first week of 2023)")

# The window must never see forwards. Checked directly rather than trusted:
# the weekly lag of a row must equal the pickup count 168 hours earlier.
lookahead = model_table.alias("now").join(
    model_table.select(
        F.col("date").alias("then_date"),
        F.col("hour").alias("then_hour"),
        F.col("airport").alias("then_airport"),
        F.col("n_pickups").alias("then_pickups"),
    ).alias("then"),
    (F.col("now.airport") == F.col("then.then_airport"))
    & (F.date_sub(F.col("now.date"), 7) == F.col("then.then_date"))
    & (F.col("now.hour") == F.col("then.then_hour")),
    how="inner",
).where(F.col("now.pickups_lag_168h") != F.col("then.then_pickups")).count()
assert lookahead == 0, f"{lookahead:,} rows have a weekly lag that is not seven days back"
print("Weekly lag verified against a date-arithmetic join.")

### The benchmark

`pickups_lag_168h` on its own is a seasonal-naive forecast: *this hour will be what it was
last week*. It is a genuinely strong predictor of airport demand and it costs nothing, so it
is the number both models have to beat. Reporting it here means notebook 4 compares against a
benchmark that was fixed before any model was fitted, and the report can say *our model beats
a sensible benchmark by X* rather than quoting an unanchored error figure.

Computed on the test split only, and excluding the six daylight-saving rows, which is the
same basis notebook 4 will use.

In [ ]:
def error_metrics(frame: DataFrame, actual: str, predicted: str) -> dict:
    """Root mean squared error and mean absolute error between two columns.

    Args:
        frame: Frame holding both columns.
        actual: Observed value column.
        predicted: Predicted value column.

    Returns:
        Mapping with ``n``, ``rmse``, and ``mae``. Rows where either column is
        null are excluded and reflected in ``n``.
    """
    error = F.col(actual) - F.col(predicted)
    row = (
        frame
        .where(F.col(actual).isNotNull() & F.col(predicted).isNotNull())
        .agg(
            F.count("*").alias("n"),
            F.sqrt(F.avg(F.pow(error, 2))).alias("rmse"),
            F.avg(F.abs(error)).alias("mae"),
        )
        .collect()[0]
    )
    return {"n": row["n"], "rmse": round(row["rmse"], 3), "mae": round(row["mae"], 3)}


test_rows = model_table.where(
    (F.col("date") >= F.lit(TRAIN_END).cast("date")) & (~F.col("dst_anomaly"))
)

baseline_pickups = error_metrics(test_rows, "n_pickups", "pickups_lag_168h")
baseline_value = error_metrics(test_rows, "mean_total", "mean_total_lag_168h")

print(f"Seasonal-naive benchmark on the test period ({TRAIN_END} onwards)")
print(f"  n_pickups   n={baseline_pickups['n']:,}  "
      f"RMSE {baseline_pickups['rmse']:.2f} pickups  "
      f"MAE {baseline_pickups['mae']:.2f}")
print(f"  mean_total  n={baseline_value['n']:,}  "
      f"RMSE ${baseline_value['rmse']:.2f}  "
      f"MAE ${baseline_value['mae']:.2f}")

## Step 5 — Enforce the leakage rule

Notebook 2b wrote `flight_column_roles.json` precisely so that this notebook could enforce
the separation rather than rely on someone remembering it. Every column under
`realised_lag_only` is an outcome of the hour it describes — how late the arrivals actually
ran, how many were cancelled — and a driver deciding at 16:00 whether to join the queue
cannot know any of it. Each is lagged by one hour and the unlagged original is excluded from
the feature list.

**The taxi side has no roles file, and needs the same treatment.** Every column of Table A
except the keys and `dst_anomaly` describes trips that have *already happened* in that hour:
`mean_fare`, `share_flat_fare`, `n_card_trips`, `mean_distance_mi`, `mean_tip_ratio`. For
Model 2 the danger is blunt — `mean_fare` in the same hour would be predicting `mean_total`
from very nearly itself, and the model would report an excellent fit for having been told the
answer. They are declared here as leak-if-unlagged, and the five with genuine predictive
value are carried as explicit one-hour lags.

One hour is the right lag for these. Weekly and daily history is already covered above; what
a one-hour lag adds is the state of the queue *just now*, which is exactly the information a
driver at the airport actually has.

In [ ]:
# --- Flight side: enforce the contract written by 2b -----------------------
realised_columns = (
    flight_roles["realised_lag_only"] + flight_roles["ewr_realised_lag_only"]
)
missing = [column for column in realised_columns if column not in model_table.columns]
assert not missing, f"Columns declared by 2b are absent from the join: {missing}"

for column in realised_columns:
    model_table = model_table.withColumn(
        f"{column}_lag_1h", F.lag(column, LAG_HOUR).over(by_hour)
    )

# --- Taxi side: the same rule, declared here ------------------------------
TAXI_KEYS = ["date", "hour", "airport"]
TAXI_FLAGS = ["dst_anomaly"]
# The two targets are same-hour outcomes too, by definition. They are named
# separately rather than listed as leaking columns, so that the manifest reads
# as "these may not be features" rather than casting suspicion on the targets.
TARGETS = ["n_pickups", "mean_total"]
taxi_same_hour = [
    column for column in table_a.columns
    if column not in TAXI_KEYS + TAXI_FLAGS + TARGETS
]

# The five worth carrying forward. The rest stay in the table for notebook 3's
# descriptive work and are barred from the feature list by the manifest.
TAXI_LAGGED = [
    "mean_fare",
    "share_flat_fare",
    "mean_distance_mi",
    "mean_tip_ratio",
    "share_flex_fare",
]
assert set(TAXI_LAGGED) <= set(taxi_same_hour), "A lagged taxi column is not in Table A"

for column in TAXI_LAGGED:
    model_table = model_table.withColumn(
        f"{column}_lag_1h", F.lag(column, LAG_HOUR).over(by_hour)
    )

leak_if_unlagged = sorted(set(taxi_same_hour + realised_columns))
print(f"{len(leak_if_unlagged)} columns declared leak-if-unlagged")
print(f"{len(realised_columns) + len(TAXI_LAGGED)} one-hour lagged features built")

## Step 6 — Materialise the split

The split is written into the table as a column rather than applied as a date filter when
each model is fitted. Two models fitted in two places from two filters is two chances for
them to drift apart, and a comparison between models trained on different data is not a
comparison. Written here, both models provably see the same split, and a marker can verify it
by reading one column instead of two notebooks.

Train is the calendar year 2023 (17,520 rows); test is January–June 2024 (8,736). The split
runs forward in time, as the specification requires: a shuffled or k-fold scheme would let
the model learn from June 2024 in order to predict February 2024, which no driver can do.

In [ ]:
model_table = model_table.withColumn(
    "split",
    F.when(F.col("date") < F.lit(TRAIN_END).cast("date"), F.lit("train"))
    .otherwise(F.lit("test")),
)

split_counts = {
    row["split"]: row["rows"]
    for row in model_table.groupBy("split").agg(F.count("*").alias("rows")).collect()
}
assert split_counts["train"] == EXPECTED_TRAIN_ROWS, split_counts
assert split_counts["test"] == EXPECTED_TEST_ROWS, split_counts
assert sum(split_counts.values()) == EXPECTED_ROWS, split_counts

print(f"train  {split_counts['train']:,} rows  (2023)")
print(f"test   {split_counts['test']:,} rows  ({TRAIN_END} to {WINDOW_END})")

## Step 7 — Validate and write

Seven checks. Beyond the row count and key uniqueness, two are worth naming.

**The decomposition identity.** The whole two-model design rests on
`n_pickups × mean_total = sum_total_amount`. If that does not hold in the data, multiplying
the two predictions does not estimate revenue and the recommendations rest on nothing. It is
checked with a tolerance rather than for equality, because a distributed sum of doubles is
not associative and may differ in the last cents.

**The null audit.** Every column is counted and each family of nulls has to be one this
notebook already documented: the first week's lags, the means of empty hours, the weather
gaps, the delay columns where nothing was observed, the neighbouring-hour columns at the ends
of the window, and `holiday_name` on ordinary days. Anything else is an unexplained null and
fails the run.

In [ ]:
# 1. The grid is intact and each key appears exactly once.
assert model_table.count() == EXPECTED_ROWS
assert model_table.dropDuplicates(["date", "hour", "airport"]).count() == EXPECTED_ROWS

# 2. No pickup was lost or duplicated by the four joins.
joined_pickups = model_table.agg(F.sum("n_pickups")).collect()[0][0]
source_pickups = table_a.agg(F.sum("n_pickups")).collect()[0][0]
assert joined_pickups == source_pickups, (
    f"{joined_pickups:,} pickups after the joins, {source_pickups:,} in Table A"
)

# 3. The decomposition the two models rely on.
identity = model_table.where(F.col("n_pickups") > 0).agg(
    F.max(
        F.abs(F.col("n_pickups") * F.col("mean_total") - F.col("sum_total_amount"))
    ).alias("max_abs_error")
).collect()[0]["max_abs_error"]
assert identity < 0.01, f"n_pickups × mean_total does not reproduce revenue: {identity}"
print(f"Revenue decomposition holds to ${identity:.6f}")

# 4. Daylight saving is still flagged on exactly its six rows.
dst_rows = model_table.where(F.col("dst_anomaly")).count()
assert dst_rows == 6, f"{dst_rows} DST rows flagged, expected 6"

# 5. Model 2's training set is smaller than Model 1's, by exactly the hours
#    with no pickups. Reported, because it is a limitation the report states.
empty_hours = model_table.where(F.col("n_pickups") == 0).count()
undefined_value = model_table.where(F.col("mean_total").isNull()).count()
assert empty_hours == undefined_value, (
    "mean_total is null in a different set of hours from the empty ones"
)
print(f"{empty_hours:,} airport-hours with no pickups — Model 2 is undefined in these")

# 6. Shares still lie in [0, 1] after the joins and lags.
for column in ["share_flat_fare", "share_longhaul", "share_cancelled"]:
    out_of_range = model_table.where(
        F.col(column).isNotNull() & ~F.col(column).between(0, 1)
    ).count()
    assert out_of_range == 0, f"{column} out of range in {out_of_range:,} rows"

print("Structural checks passed.")

In [ ]:
# 7. The null audit. Every column with nulls must belong to a documented family.
null_counts = {
    column: count
    for column, count in model_table.select([
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in model_table.columns
    ]).collect()[0].asDict().items()
    if count
}

# Families, in the order they are tested. A column is explained by the first
# family it matches.
LAG_SUFFIXES = ("_lag_1h", "_lag_24h", "_lag_168h", "_same_hour_7d")
lagged_columns = [
    column for column in model_table.columns if column.endswith(LAG_SUFFIXES)
]
empty_hour_means = [
    "mean_fare", "median_fare", "mean_total", "mean_distance_mi",
    "mean_duration_min", "mean_passengers", "share_flat_fare", "mean_tip_ratio",
]
weather_columns = [
    "temp_c", "rhum_pct", "prcp_mm", "wspd_kmh", "pres_hpa", "coco_code",
    "condition", "is_precip", "is_snow",
]
nothing_scheduled_columns = [
    "share_longhaul", "share_cancelled", "ewr_share_cancelled",
]
delay_columns = [
    "mean_arr_delay_min", "share_delayed_15", "ewr_mean_arr_delay_min",
]
window_edge_columns = ["sched_arrivals_prev_hr", "sched_arrivals_next_hr"]

FAMILIES = [
    ("first-week lags and one-hour lags", lagged_columns),
    ("statistic over an empty subset of the hour's trips", empty_hour_means),
    ("weather gaps beyond the fill limit", weather_columns),
    ("nothing scheduled to land in the hour", nothing_scheduled_columns),
    ("no delay observed in the hour", delay_columns),
    ("first and last hour of the window", window_edge_columns),
    ("ordinary (non-holiday) days", ["holiday_name"]),
]

unexplained = []
print(f"{'column':<30} {'nulls':>8}  family")
for column, count in sorted(null_counts.items(), key=lambda item: -item[1]):
    family = next(
        (name for name, members in FAMILIES if column in members), None
    )
    print(f"{column:<30} {count:>8,}  {family or '*** UNEXPLAINED ***'}")
    if family is None:
        unexplained.append(column)

assert not unexplained, f"Undocumented nulls in: {unexplained}"
print("\nNull audit passed.")

### The roles manifest

`model_table_roles.json` is to notebook 4 what `flight_column_roles.json` was to this one: a
feature list as data rather than as a comment, so the modelling notebook reads it instead of
hardcoding twenty column names that then drift.

Features are grouped by kind because the two models want different subsets. The Poisson GLM
takes the cyclical hour terms and needs its categoricals indexed; the trees take the raw hour
and ignore the sine and cosine. Both take the flight, weather, and lag blocks.

`flags` are not features. `dst_anomaly` and `has_full_lags` exist so that notebook 4 can
exclude those rows; `weather_imputed` and `weather_missing` record how much of the weather
was observed rather than carried forward.

In [ ]:
FEATURES = {
    "temporal": ["hour", "day_of_week", "month", "is_weekend",
                 "is_holiday", "is_holiday_adjacent"],
    "cyclical": ["hour_sin", "hour_cos"],
    "categorical": ["airport", "day_of_week", "month", "condition"],
    "flight_schedule": flight_roles["features"],
    "flight_realised_lagged": [
        f"{column}_lag_1h" for column in flight_roles["realised_lag_only"]
    ],
    "ewr": flight_roles["ewr_features"] + [
        f"{column}_lag_1h" for column in flight_roles["ewr_realised_lag_only"]
    ],
    "weather": ["temp_c", "rhum_pct", "prcp_mm", "wspd_kmh", "pres_hpa",
                "condition", "is_precip", "is_snow"],
    "taxi_lagged": [f"{column}_lag_1h" for column in TAXI_LAGGED],
    "lags_volume": ["pickups_lag_1h", "pickups_lag_24h", "pickups_lag_168h",
                    "pickups_same_hour_7d"],
    "lags_value": ["mean_total_lag_168h", "mean_total_same_hour_7d"],
}

ROLES = {
    "key": ["date", "hour", "airport"],
    "rows": EXPECTED_ROWS,
    "targets": {
        "model_1_volume": "n_pickups",
        "model_2_value": "mean_total",
        "product": "expected revenue per airport-hour; reconciles with sum_total_amount",
    },
    "features": FEATURES,
    "benchmark": {
        "volume": "pickups_lag_168h",
        "value": "mean_total_lag_168h",
        "test_rmse_pickups": baseline_pickups["rmse"],
        "test_mae_pickups": baseline_pickups["mae"],
        "test_rmse_mean_total": baseline_value["rmse"],
        "test_mae_mean_total": baseline_value["mae"],
    },
    "flags": ["dst_anomaly", "has_full_lags", "weather_imputed", "weather_missing"],
    "leak_if_unlagged": leak_if_unlagged,
    "split": {
        "column": "split",
        "train": f"{WINDOW_START} to {TRAIN_END} (exclusive)",
        "test": f"{TRAIN_END} to {WINDOW_END} (exclusive)",
        "train_rows": EXPECTED_TRAIN_ROWS,
        "test_rows": EXPECTED_TEST_ROWS,
    },
    "note": (
        "Columns under leak_if_unlagged describe trips or arrivals that have "
        "already happened in the hour they key. They may enter a model only at "
        "a lag of at least one hour. Nothing in this table is scaled or "
        "centred: standardisation belongs inside the model fit, on the "
        "training split alone. Rows where has_full_lags is false, and rows "
        "where dst_anomaly is true, should be excluded from the fit."
    ),
}

# Every declared feature must exist, and no feature may be a leaking column.
declared = [column for group in FEATURES.values() for column in group]
absent = [column for column in declared if column not in model_table.columns]
assert not absent, f"Declared features absent from the table: {absent}"

leaking = sorted(set(declared) & set(leak_if_unlagged))
assert not leaking, f"Unlagged same-hour outcomes in the feature list: {leaking}"

with open(CURATED_DIR / "model_table_roles.json", "w") as handle:
    json.dump(ROLES, handle, indent=2)

print(f"{len(set(declared))} distinct features declared across {len(FEATURES)} groups")
print("No same-hour outcome reaches the feature list.")

In [ ]:
(
    model_table
    .orderBy("date", "hour", "airport")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(CURATED_DIR / "model_table.parquet"))
)

weather_quality = model_table.agg(
    F.sum(F.col("weather_imputed").cast("int")).alias("filled"),
    F.sum(F.col("weather_missing").cast("int")).alias("missing"),
).collect()[0]

shapes = {
    "model_table_rows": EXPECTED_ROWS,
    "model_table_columns": len(model_table.columns),
    "features_declared": len(set(declared)),
    "train_rows": split_counts["train"],
    "test_rows": split_counts["test"],
    "rows_without_full_lags": incomplete,
    "zero_pickup_hours": empty_hours,
    "hours_model_2_undefined": undefined_value,
    "weather_hours_carried_forward": weather_quality["filled"],
    "weather_hours_unobserved": weather_quality["missing"],
    "benchmark_test_rmse_pickups": baseline_pickups["rmse"],
    "benchmark_test_mae_pickups": baseline_pickups["mae"],
    "benchmark_test_rmse_mean_total": baseline_value["rmse"],
    "benchmark_test_mae_mean_total": baseline_value["mae"],
}
with open(CURATED_DIR / "shapes_model_table.json", "w") as handle:
    json.dump(shapes, handle, indent=2)

shapes

In [ ]:
model_table.select(
    "date", "hour", "airport", "n_pickups", "mean_total", "sched_arrivals",
    "temp_c", "condition", "pickups_lag_168h", "split",
).orderBy("date", "hour", "airport").show(8, truncate=False)

In [ ]:
spark.stop()

## What to carry into the report

- **The two-model design is the story of the modelling section.** One sentence in the
  introduction's methodology overview: demand is modelled as a count, value as a continuous
  amount, and their product is expected revenue per airport-hour. The three marks for
  combining two models are earned by that product, not by fitting two things.
- **The benchmark.** Quote the seasonal-naive RMSE and MAE from Step 4 and compare every
  model against them. A model that fails to beat *this hour will be what it was last week* is
  a finding worth reporting honestly, not one to bury.
- **The leakage argument.** Two sentences: a driver deciding at 16:00 knows the published
  arrivals board and does not know how late the flights will run or what the last hour's
  fares were, and the roles files enforce that separation in code rather than by convention.
  This is the single easiest place in the project to lose marks silently and the easiest to
  defend once stated.
- **Weather preprocessing.** The dropped columns (`snow`, `wpgt`) with their null rates, the
  two fill policies and why they differ, and the Central Park cross-check. That is the second
  external dataset's preprocessing dot points, and the specification asks for shapes at each
  step.
- **The holiday constant** with its OPM citation, and the note that observed dates are
  included alongside actual ones.
- **Model 2's smaller support.** The hours where `mean_total` is undefined are the empty
  overnight hours, so the value model is silent exactly where the demand model says not to
  go. State it as a limitation rather than letting a marker find it.
- **Nothing here is scaled.** If notebook 4 standardises for the GLM, it fits the scaler on
  the training split alone and says so.

**Next:** notebook 3 reads `trips_clean.parquet` and `model_table.parquet` for the
distribution, outlier, and geospatial work; notebook 4 reads `model_table.parquet` and
`model_table_roles.json` and fits the two models.